In [ ]:
%load_ext autoreload
%autoreload 2


# Klone/HYAK vLLM Ops

A short path from a laptop to a Qwen3-8B vLLM endpoint running on a Klone GPU node. The laptop runs the client; Klone runs only the GPU server behind an SSH tunnel.


## Install the local package

Run this from the repo root so the notebook can import `slurm_ops`:

```bash
uv sync
```

The exported module for this workflow is `slurm_ops.vllm`, sourced from `nbs/02_vllm.ipynb`.


In [ ]:
from slurm_ops.vllm import make_slurm_args, vllm_up, vllm_chat, vllm_down


## One-time SSH setup

Use a `klone-login` SSH alias that points at `klone.hyak.uw.edu`. ControlMaster keeps the Duo-authenticated connection around so the Python helpers can run many SSH commands without prompting again.

```sshconfig
Host klone-login
    HostName klone.hyak.uw.edu
    User <your-netid>
    ControlMaster auto
    ControlPath ~/.ssh/cm-%r@%h:%p
    ControlPersist 10h
    ServerAliveInterval 60
```

Seed the connection once per working session:

```bash
ssh klone-login true
```


## Sync the cluster-side vLLM files

`vllm_up` expects `serve.sh`, `vllm.def`, and `build-sif.job` under `~/slurm-ops/vllm` on Klone.

```bash
ssh klone-login 'mkdir -p ~/slurm-ops/vllm'
rsync -a vllm/ klone-login:slurm-ops/vllm/
ssh klone-login 'chmod +x ~/slurm-ops/vllm/*.sh ~/slurm-ops/vllm/*.job ~/slurm-ops/vllm/bin/*'
```

The default SIF path is `/mmfs1/gscratch/scrubbed/$USER/vllm.sif`, with a shared fallback at `/mmfs1/gscratch/scrubbed/aurasoph/vllm.sif` when readable.


## Start Qwen on Klone

The public default is the `stf` account on `gpu-l40s` for four hours:

```python
info = vllm_up("qwen", "klone-login")
```

For local testing with the `amath` account, override only the resource string:

```python
info = vllm_up(
    "qwen-test",
    "klone-login",
    slurm_args=make_slurm_args(account="amath", time_limit="01:00:00"),
)
```

The call blocks until `/v1/models` is live and returns `base_url`, `served_name`, the compute node, and ports.


In [ ]:
# Public default: account=stf, partition=gpu-l40s, time=04:00:00
# info = vllm_up("qwen", "klone-login")

# Local test override for users with amath access:
# info = vllm_up("qwen-test", "klone-login", slurm_args=make_slurm_args(account="amath", time_limit="01:00:00"))


## Ask Qwen something

`vllm_chat` uses the OpenAI-compatible HTTP endpoint directly; it does not need the OpenAI Python package.


In [ ]:
# vllm_chat("Reply with exactly: qwen-ready", base_url=info["base_url"], model=info["served_name"])


## Shell equivalent

```bash
./vllm/bin/vllm-up qwen klone-login
./vllm/bin/vllm-chat --base-url http://localhost:8000/v1 "Reply with exactly: qwen-ready"
./vllm/bin/vllm-down qwen klone-login --local-port 8000
```

Testing with `amath`:

```bash
./vllm/bin/vllm-up qwen-test klone-login --account amath --time 01:00:00
./vllm/bin/vllm-chat --base-url http://localhost:8000/v1 "Reply with exactly: qwen-ready"
./vllm/bin/vllm-down qwen-test klone-login --local-port 8000
```

If port 8000 is busy, use the `base_url`, chat command, and stop command printed by `vllm-up`.


## Debugging

```bash
ssh klone-login -t tmux attach -t qwen
ssh klone-login tail -f ~/.vllm-discovery/qwen.log
ssh klone-login cat ~/.vllm-discovery/qwen.json
ssh klone-login squeue --me
```
